# Nemotron Path A · Tinker-Adapter Packaging (V7)

**WayneIA / bbobwayne** · 2026-05-10 · per `PPWHH_WAYNEIQ_TO_P1_NEMOTRON_OCA_THROWING_IN_v2_20260510.md`

**Path A · USE_PRETRAINED Mode B**: package public Kaggle adapter for submission · zero training · zero compute spend.

## Provenance + credit

This package consumes the public Kaggle model `kienngx/nemotron-nano-30b-trained/Triton/tinker-adapter/1` (Ngô Xuân Kiên). The same source produces a 0.86 public-LB anchor verified across multiple independent public kernels: `afr1ste/nemotron-0-86-tinker-adapter-guide`, `mohamedamr992/0-86-adapter-packaging-workflow`, `xduan7/nemotron-dgxchen-repro-eval`. Credit to the original publisher.

## Audit checklist (verified)
- adapter rank `r=32` (max_lora_rank=32 compliant)
- adapter_config.json + adapter_model.safetensors both present at zip root
- base_model = `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16`
- `enable_gpu=false` · packaging-only · no Pro-6000 allocation gate
- `enable_internet=false`

Replaces V6 heuristic-CSV approach (ERRORed: Code Comp evaluator expects LoRA adapter zip, not CSV).

In [1]:
# Cell 1 · Locate source adapter (Pattern #19 substrate-probe-FIRST)
from pathlib import Path
import json, os

INPUT_ROOT = Path('/kaggle/input')
REQUIRED = ['adapter_config.json', 'adapter_model.safetensors']

candidates = []
for config_path in INPUT_ROOT.rglob('adapter_config.json'):
    adapter_dir = config_path.parent
    if all((adapter_dir / name).exists() for name in REQUIRED):
        size = (adapter_dir / 'adapter_model.safetensors').stat().st_size
        candidates.append((adapter_dir, size))

assert candidates, 'No adapter directory found under /kaggle/input · check kernel-metadata.json model_sources'

# Prefer tinker-adapter path (most-vetted 0.86 baseline)
candidates.sort(key=lambda item: ('tinker' not in str(item[0]).lower(), -item[1]))
for i, (path, size) in enumerate(candidates, start=1):
    print(f'{i:02d}. {path} · {size / (1024**3):.3f} GiB')

ADAPTER_DIR = candidates[0][0]
print(f'\nSelected: {ADAPTER_DIR}')

# Inspect adapter_config
cfg = json.loads((ADAPTER_DIR / 'adapter_config.json').read_text())
print(f'peft_type={cfg.get("peft_type")} · r={cfg.get("r")} · lora_alpha={cfg.get("lora_alpha")}')
print(f'base_model={cfg.get("base_model_name_or_path")}')
assert cfg.get('r', 0) <= 32, f'max_lora_rank=32 violated (r={cfg.get("r")})'
print('Audit PASS · rank <= 32 · LoRA peft_type · base_model present')

01. /kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1 · 3.310 GiB

Selected: /kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1
peft_type=LORA · r=32 · lora_alpha=32
base_model=nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
Audit PASS · rank <= 32 · LoRA peft_type · base_model present


In [2]:
# Cell 2 · Package submission.zip (adapter_config.json + adapter_model.safetensors at zip ROOT)
import shutil, zipfile, os
from pathlib import Path

WORKING = Path('/kaggle/working')
STAGE = WORKING / 'submission_stage'
STAGE.mkdir(exist_ok=True)

# Copy required files to stage
for fname in ['adapter_config.json', 'adapter_model.safetensors']:
    src = ADAPTER_DIR / fname
    dst = STAGE / fname
    shutil.copy(src, dst)
    print(f'staged: {fname} · {dst.stat().st_size} bytes')

# Create submission.zip with files at root (ZIP_STORED for max safetensors compat)
ZIP_PATH = WORKING / 'submission.zip'
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_STORED) as zf:
    for fname in ['adapter_config.json', 'adapter_model.safetensors']:
        zf.write(STAGE / fname, arcname=fname)

# Verify zip layout
with zipfile.ZipFile(ZIP_PATH) as zf:
    names = zf.namelist()
    print(f'\nsubmission.zip contents: {names}')
    assert 'adapter_config.json' in names and 'adapter_model.safetensors' in names
    print(f'submission.zip size: {ZIP_PATH.stat().st_size / (1024**3):.3f} GiB')

print('\nDone · /kaggle/working/submission.zip ready for `kaggle competitions submit -f submission.zip`')

staged: adapter_config.json · 581 bytes
staged: adapter_model.safetensors · 3554384888 bytes

submission.zip contents: ['adapter_config.json', 'adapter_model.safetensors']
submission.zip size: 3.310 GiB

Done · /kaggle/working/submission.zip ready for `kaggle competitions submit -f submission.zip`
